# Experiment 4: Industry Effects on Salary and Skill Requirements

**Hypothesis:**  
*Job postings in technology and finance industries will exhibit higher average salaries and more advanced skill requirements than other sectors.*

This notebook follows the same workflow style as Experiment 2, reuses functions from `helpers.py`, and tests the hypothesis using:
1. **Salary comparison** across industry groups  
2. **Advanced-skill requirement comparison** across industry groups  
3. **A final support / mixed / reject conclusion**


In [ ]:
!pip install pandas
!pip install matplotlib
!pip install scipy

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from helpers import parse_salary, extract_skill_lists, SKILL_PATTERNS

# Try to use the same unified dataset builder pattern as Experiment 2.
try:
    from job_dataset_union import build_unified_jobs_df
    df = build_unified_jobs_df()
    data_source_note = "Loaded data using build_unified_jobs_df() from job_dataset_union.py"
except Exception as e:
    # Fallback: try common local CSV/parquet names if job_dataset_union is unavailable.
    candidate_files = [
        "unified_jobs.csv", "merged_jobs.csv", "jobs_unified.csv",
        "unified_jobs.parquet", "merged_jobs.parquet", "jobs_unified.parquet"
    ]
    loaded = False
    for fname in candidate_files:
        path = Path(fname)
        if path.exists():
            if path.suffix == ".csv":
                df = pd.read_csv(path)
            else:
                df = pd.read_parquet(path)
            data_source_note = f"Loaded fallback dataset from {fname}"
            loaded = True
            break

    if not loaded:
        raise FileNotFoundError(
            "Could not load the unified jobs dataset. "
            "Please place `job_dataset_union.py` next to this notebook or add a fallback unified CSV/parquet file."
        ) from e

print(data_source_note)
print(f"Loaded {len(df)} job postings.")
print("\nColumns:")
print(df.columns.tolist())

In [ ]:
# Identify likely columns for job descriptions, salary, and industry
job_desc_candidates = ["job_description_skills", "job_description", "description", "Job Description"]
salary_candidates = ["salary_range", "Salary Estimate", "salary", "pay_range"]
industry_candidates = ["industry", "Industry", "industry_name", "sector", "Sector", "company_industry"]

job_desc_col = next((c for c in job_desc_candidates if c in df.columns), None)
salary_col = next((c for c in salary_candidates if c in df.columns), None)
industry_col = next((c for c in industry_candidates if c in df.columns), None)

if job_desc_col is None:
    raise ValueError(f"Could not find a job description column. Checked: {job_desc_candidates}")
if salary_col is None:
    raise ValueError(f"Could not find a salary column. Checked: {salary_candidates}")
if industry_col is None:
    raise ValueError(f"Could not find an industry/sector column. Checked: {industry_candidates}")

print(f"Using job description column: {job_desc_col}")
print(f"Using salary column: {salary_col}")
print(f"Using industry column: {industry_col}")

print("\nTop raw industry values:")
print(df[industry_col].fillna("Missing").astype(str).value_counts().head(20))

In [ ]:
# Parse salary and keep clean text columns
df = df.copy()
df["Salary_Numeric"] = df[salary_col].apply(parse_salary)
df["job_text"] = df[job_desc_col].fillna("").astype(str)
df["industry_text"] = df[industry_col].fillna("Unknown").astype(str)

df_valid = df[df["Salary_Numeric"].notna()].copy()

print(f"Rows with valid salary data: {len(df_valid)} / {len(df)}")
print("\nSalary summary ($):")
print(df_valid["Salary_Numeric"].describe())
print("\nSalary summary ($K):")
print((df_valid["Salary_Numeric"] / 1000).describe())

In [ ]:
# Group industries into Technology, Finance, and Other
TECH_PATTERN = re.compile(
    r"software|information technology|it services|internet|computer|semiconductor|"
    r"electronics|cyber|cloud|saas|data platform|telecommunications|tech",
    flags=re.IGNORECASE
)

FINANCE_PATTERN = re.compile(
    r"financial|bank|banking|investment|capital markets|asset management|insurance|"
    r"fintech|payments|trading|wealth management|credit|lending|private equity|hedge fund",
    flags=re.IGNORECASE
)

def classify_industry_group(x: str) -> str:
    x = str(x)
    if TECH_PATTERN.search(x):
        return "Technology"
    elif FINANCE_PATTERN.search(x):
        return "Finance"
    else:
        return "Other"

df_valid["Industry_Group"] = df_valid["industry_text"].apply(classify_industry_group)

print(df_valid["Industry_Group"].value_counts())
print("\nSample mapped industries:")
print(
    df_valid[[industry_col, "Industry_Group"]]
    .drop_duplicates()
    .head(25)
    .to_string(index=False)
)

In [ ]:
# Define "advanced skills" using the shared helper skill library
ADVANCED_SKILLS = [
    "Machine Learning", "Deep Learning", "NLP", "Computer Vision", "Generative AI",
    "Reinforcement Learning", "Scikit-learn", "TensorFlow", "PyTorch", "XGBoost",
    "LightGBM", "Spark", "Databricks", "AWS", "Azure", "GCP", "Docker",
    "Kubernetes", "Airflow", "dbt", "Snowflake", "BigQuery", "Model Deployment",
    "Feature Engineering", "Data Warehousing"
]

skill_lists = extract_skill_lists(df_valid["job_text"], SKILL_PATTERNS)
df_valid["skill_list"] = skill_lists
df_valid["advanced_skill_count"] = df_valid["skill_list"].apply(
    lambda skills: sum(skill in ADVANCED_SKILLS for skill in skills)
)
df_valid["has_advanced_skill"] = df_valid["advanced_skill_count"] > 0

print(df_valid[["Industry_Group", "advanced_skill_count", "Salary_Numeric"]].head())

In [ ]:
# Industry-level summary table
group_summary = (
    df_valid.groupby("Industry_Group")
    .agg(
        Postings=("Industry_Group", "size"),
        Avg_Salary=("Salary_Numeric", "mean"),
        Median_Salary=("Salary_Numeric", "median"),
        Avg_Advanced_Skills=("advanced_skill_count", "mean"),
        Median_Advanced_Skills=("advanced_skill_count", "median"),
        Share_With_Advanced_Skills=("has_advanced_skill", "mean"),
    )
    .reset_index()
)

group_summary["Avg_Salary_K"] = group_summary["Avg_Salary"] / 1000
group_summary["Median_Salary_K"] = group_summary["Median_Salary"] / 1000
group_summary["Share_With_Advanced_Skills"] = group_summary["Share_With_Advanced_Skills"] * 100

group_summary = group_summary[
    ["Industry_Group", "Postings", "Avg_Salary_K", "Median_Salary_K",
     "Avg_Advanced_Skills", "Median_Advanced_Skills", "Share_With_Advanced_Skills"]
].sort_values("Avg_Salary_K", ascending=False)

print("Industry comparison summary:")
print(group_summary.to_string(index=False, float_format=lambda x: f"{x:,.2f}"))

In [ ]:
# Statistical testing: compare Technology+Finance vs Other
from scipy.stats import ttest_ind, mannwhitneyu

target = df_valid[df_valid["Industry_Group"].isin(["Technology", "Finance"])].copy()
other = df_valid[df_valid["Industry_Group"] == "Other"].copy()

print(f"Technology + Finance postings: {len(target)}")
print(f"Other postings: {len(other)}")

salary_t = ttest_ind(target["Salary_Numeric"], other["Salary_Numeric"], equal_var=False, nan_policy="omit")
skill_t = ttest_ind(target["advanced_skill_count"], other["advanced_skill_count"], equal_var=False, nan_policy="omit")

salary_u = mannwhitneyu(target["Salary_Numeric"], other["Salary_Numeric"], alternative="two-sided")
skill_u = mannwhitneyu(target["advanced_skill_count"], other["advanced_skill_count"], alternative="two-sided")

print("Welch t-test for salary:")
print(salary_t)
print("\nWelch t-test for advanced skill count:")
print(skill_t)
print("\nMann-Whitney U test for salary:")
print(salary_u)
print("\nMann-Whitney U test for advanced skill count:")
print(skill_u)

In [ ]:
# Separate look at Technology and Finance versus Other
tech = df_valid[df_valid["Industry_Group"] == "Technology"]
finance = df_valid[df_valid["Industry_Group"] == "Finance"]

comparison_rows = []

for label, subset in [("Technology", tech), ("Finance", finance)]:
    if len(subset) > 1 and len(other) > 1:
        salary_test = ttest_ind(subset["Salary_Numeric"], other["Salary_Numeric"], equal_var=False, nan_policy="omit")
        skill_test = ttest_ind(subset["advanced_skill_count"], other["advanced_skill_count"], equal_var=False, nan_policy="omit")
        comparison_rows.append({
            "Group": label,
            "N": len(subset),
            "Avg Salary ($K)": subset["Salary_Numeric"].mean() / 1000,
            "Other Avg Salary ($K)": other["Salary_Numeric"].mean() / 1000,
            "Salary p-value": salary_test.pvalue,
            "Avg Advanced Skills": subset["advanced_skill_count"].mean(),
            "Other Avg Advanced Skills": other["advanced_skill_count"].mean(),
            "Advanced Skills p-value": skill_test.pvalue,
        })

comparison_df = pd.DataFrame(comparison_rows)
print(comparison_df.to_string(index=False, float_format=lambda x: f"{x:,.4f}"))

In [ ]:
# Visualization 1: average salary by industry group
salary_plot_df = group_summary.sort_values("Avg_Salary_K", ascending=False)

plt.figure(figsize=(9, 6))
plt.bar(salary_plot_df["Industry_Group"], salary_plot_df["Avg_Salary_K"], alpha=0.85)
plt.ylabel("Average Salary ($K)")
plt.xlabel("Industry Group")
plt.title("Average Salary by Industry Group")
for i, v in enumerate(salary_plot_df["Avg_Salary_K"]):
    plt.text(i, v + 1, f"${v:.1f}K", ha="center")
plt.tight_layout()
plt.show()

In [ ]:
# Visualization 2: average advanced skill count by industry group
skill_plot_df = group_summary.sort_values("Avg_Advanced_Skills", ascending=False)

plt.figure(figsize=(9, 6))
plt.bar(skill_plot_df["Industry_Group"], skill_plot_df["Avg_Advanced_Skills"], alpha=0.85)
plt.ylabel("Average Advanced Skill Count")
plt.xlabel("Industry Group")
plt.title("Advanced Skill Requirements by Industry Group")
for i, v in enumerate(skill_plot_df["Avg_Advanced_Skills"]):
    plt.text(i, v + 0.03, f"{v:.2f}", ha="center")
plt.tight_layout()
plt.show()

In [ ]:
# Most common advanced skills inside each industry group
rows = []
for group_name, subset in df_valid.groupby("Industry_Group"):
    for skill in ADVANCED_SKILLS:
        pct = subset["skill_list"].apply(lambda skills: skill in skills).mean() * 100
        rows.append({
            "Industry_Group": group_name,
            "Skill": skill,
            "Pct_Postings": pct
        })

industry_skill_df = pd.DataFrame(rows)

for group_name in ["Technology", "Finance", "Other"]:
    print("\n" + "="*60)
    print(f"Top advanced skills in {group_name}")
    print("="*60)
    top_skills = (
        industry_skill_df[industry_skill_df["Industry_Group"] == group_name]
        .sort_values("Pct_Postings", ascending=False)
        .head(10)
        .reset_index(drop=True)
    )
    print(top_skills.to_string(index=False, float_format=lambda x: f"{x:,.2f}"))


In [ ]:
# Final interpretation: support / mixed / reject
target_avg_salary = target["Salary_Numeric"].mean() / 1000
other_avg_salary = other["Salary_Numeric"].mean() / 1000

target_avg_skills = target["advanced_skill_count"].mean()
other_avg_skills = other["advanced_skill_count"].mean()

salary_higher = target_avg_salary > other_avg_salary
skills_higher = target_avg_skills > other_avg_skills
salary_sig = salary_t.pvalue < 0.05
skills_sig = skill_t.pvalue < 0.05

print("="*70)
print("EXPERIMENT 4 CONCLUSION")
print("="*70)

print(f"Technology + Finance average salary: ${target_avg_salary:.2f}K")
print(f"Other industries average salary:    ${other_avg_salary:.2f}K")
print(f"Technology + Finance avg advanced skills: {target_avg_skills:.2f}")
print(f"Other industries avg advanced skills:     {other_avg_skills:.2f}")
print(f"Salary p-value: {salary_t.pvalue:.4g}")
print(f"Advanced skills p-value: {skill_t.pvalue:.4g}")

if salary_higher and skills_higher and salary_sig and skills_sig:
    print("\nResult: The hypothesis is SUPPORTED.")
    print("Technology and finance postings show both significantly higher salaries and significantly more advanced skill requirements than other sectors.")
elif salary_higher and skills_higher:
    print("\nResult: The hypothesis is PARTIALLY SUPPORTED.")
    print("Technology and finance postings are directionally higher on both salary and advanced skill requirements, but at least one result is not statistically significant.")
else:
    print("\nResult: The hypothesis is NOT SUPPORTED.")
    print("The data does not show that technology and finance postings are higher on both salary and advanced skill requirements compared with other sectors.")